# LIBERO-Spatial Dataset & SmolVLA Architecture Inspection

Exploratory script that empirically verifies all architectural claims in the report.
Loads the LIBERO dataset and SmolVLA checkpoints, inspects parameter counts,
layer structure, action dimensions, and config differences between base and reference.

**GPU:** T4 or L4 (no training, just model loading + dataset inspection)  
**Runtime:** ~15 minutes (dominated by downloads)

## 0. Setup

In [ ]:
# Mount Drive for saving figures
from google.colab import drive
drive.mount('/content/drive')

import os
FIGURE_DIR = '/content/drive/MyDrive/smolvla_project/figures'
os.makedirs(FIGURE_DIR, exist_ok=True)
os.environ['MUJOCO_GL'] = 'egl'

In [ ]:
# Install dependencies
!pip install "lerobot[libero] @ git+https://github.com/huggingface/lerobot.git" -q
!pip install num2words -q

# Fix egl_probe cmake issue
!git clone https://github.com/StanfordVL/egl_probe.git /tmp/egl_probe 2>/dev/null || true
!cd /tmp/egl_probe && sed -i 's/cmake_minimum_required(VERSION 2.8.12)/cmake_minimum_required(VERSION 3.10)/' egl_probe/CMakeLists.txt && pip install . -q 2>/dev/null || true

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import json

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Dataset Exploration

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset("HuggingFaceVLA/libero")

# Inspect a single sample
sample = ds[0]
print("SAMPLE STRUCTURE")
print("=" * 60)
for key, val in sample.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.shape} ({val.dtype})")
    else:
        print(f"  {key}: {type(val).__name__} = {val}")

In [ ]:
# All tasks in the dataset
print("ALL TASKS")
print("=" * 60)
print(ds.meta.tasks)

In [ ]:
# Filter to LIBERO-Spatial (tasks involving "pick up the black bowl")
episodes_df = ds.meta.episodes.to_pandas()
spatial_mask = episodes_df["tasks"].apply(
    lambda x: any("pick up the black bowl" in t for t in x)
    if isinstance(x, list) else "pick up the black bowl" in str(x)
)
spatial_eps = episodes_df[spatial_mask]
spatial_lengths = spatial_eps["length"].tolist()

print("LIBERO-SPATIAL SUBSET")
print("=" * 60)
print(f"Total episodes in dataset: {len(episodes_df)}")
print(f"LIBERO-Spatial episodes:   {len(spatial_eps)}")
print(f"\nEpisode length stats (Spatial only):")
print(f"  Mean:   {np.mean(spatial_lengths):.1f}")
print(f"  Median: {np.median(spatial_lengths):.1f}")
print(f"  Min:    {min(spatial_lengths)}")
print(f"  Max:    {max(spatial_lengths)}")
print(f"  Std:    {np.std(spatial_lengths):.1f}")

# Per-task demo counts
task_counts = Counter()
task_lengths_map = {}
for _, row in spatial_eps.iterrows():
    name = row["tasks"]
    if isinstance(name, np.ndarray):
        name = name[0]
    task_counts[name] += 1
    task_lengths_map.setdefault(name, []).append(row["length"])

print(f"\nPer-task breakdown:")
for i, (task, count) in enumerate(sorted(task_counts.items())):
    mean_len = np.mean(task_lengths_map[task])
    print(f'  Task {i}: {count} demos, mean_len={mean_len:.0f} | "{task}"')
print(f"  Total: {sum(task_counts.values())} demos")

In [ ]:
# Action space statistics (sampled from full dataset for speed)
hf_ds = ds.hf_dataset
n_samples = min(5000, len(hf_ds))
indices = np.linspace(0, len(hf_ds) - 1, n_samples, dtype=int)
actions = np.array([np.array(hf_ds[int(i)]["action"]) for i in indices])

dim_names = ["x", "y", "z", "rx", "ry", "rz", "gripper"]
print("ACTION SPACE")
print("=" * 60)
print(f"Shape per timestep: {actions.shape[1:]}")
print(f"{'Dim':<10} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8}")
for i, name in enumerate(dim_names):
    col = actions[:, i]
    print(f"{name:<10} {col.min():>8.4f} {col.max():>8.4f} {col.mean():>8.4f} {col.std():>8.4f}")
print(f"\nRange: [{actions.min():.4f}, {actions.max():.4f}]")

In [ ]:
# Episode length histogram (Spatial only)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(spatial_lengths, bins=25, edgecolor="black", alpha=0.7, color="#3498DB")
ax.set_xlabel("Episode Length (steps)")
ax.set_ylabel("Count")
ax.set_title(f"LIBERO-Spatial Episode Length Distribution (N={len(spatial_lengths)})")
ax.axvline(np.mean(spatial_lengths), color="red", linestyle="--", lw=2,
           label=f"Mean: {np.mean(spatial_lengths):.0f}")
ax.axvline(np.median(spatial_lengths), color="orange", linestyle="--", lw=2,
           label=f"Median: {np.median(spatial_lengths):.0f}")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "episode_lengths.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved episode_lengths.png")

## 2. SmolVLA Model Config

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

model = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
config = model.config

print("KEY CONFIG VALUES")
print("=" * 60)
for field in [
    "chunk_size", "n_action_steps", "max_action_dim", "num_steps",
    "expert_width_multiplier", "num_vlm_layers", "n_obs_steps",
    "vlm_model_name", "self_attn_every_n_layers", "resize_imgs_with_padding",
    "train_expert_only", "train_state_proj", "normalization_mapping",
]:
    print(f"  {field}: {getattr(config, field, 'N/A')}")

# Default embodiment shapes (SO100)
print(f"\nBase input_features:")
for k, v in config.input_features.items():
    print(f"  {k}: {v}")
print(f"Base output_features:")
for k, v in config.output_features.items():
    print(f"  {k}: {v}")

## 3. Parameter Breakdown

In [ ]:
component_params = defaultdict(lambda: {"total": 0, "trainable": 0})
for name, param in model.named_parameters():
    if "vision" in name.lower() or "siglip" in name.lower():
        comp = "SigLIP Vision Encoder"
    elif "connector" in name.lower():
        comp = "Connector (patch merge)"
    elif "lm_head" in name.lower():
        comp = "lm_head (unused)"
    elif "lm_expert" in name.lower() or "action_expert" in name.lower():
        comp = "Action Expert"
    elif "embed" in name.lower():
        comp = "Embeddings"
    elif any(k in name.lower() for k in ["state_proj", "action_in", "action_out", "time_mlp", "time_proj"]):
        comp = "Projection layers"
    elif "language_model" in name.lower() or "text_model" in name.lower():
        comp = "SmolLM2 (language)"
    else:
        comp = "Other"

    component_params[comp]["total"] += param.numel()
    if param.requires_grad:
        component_params[comp]["trainable"] += param.numel()

total_all = sum(c["total"] for c in component_params.values())
trainable_all = sum(c["trainable"] for c in component_params.values())

print("PARAMETER BREAKDOWN")
print("=" * 60)
print(f"{'Component':<28} {'Total':>10} {'Trainable':>10} {'%':>6}")
print("-" * 58)
for comp in sorted(component_params.keys()):
    c = component_params[comp]
    pct = 100 * c["trainable"] / c["total"] if c["total"] > 0 else 0
    print(f"{comp:<28} {c['total']/1e6:>8.2f}M {c['trainable']/1e6:>8.2f}M {pct:>5.1f}%")
print("-" * 58)
print(f"{'TOTAL':<28} {total_all/1e6:>8.2f}M {trainable_all/1e6:>8.2f}M {100*trainable_all/total_all:>5.1f}%")

## 4. Action Expert Layer Structure

In [ ]:
expert_layers = model.model.vlm_with_expert.lm_expert.layers

print(f"ACTION EXPERT — {len(expert_layers)} LAYERS")
print("=" * 60)
print(f"{'Idx':>4} {'Type':<14} {'k_in':>6} {'k_out':>6} {'Params':>10}")
print("-" * 46)
for i, layer in enumerate(expert_layers):
    k_in = layer.self_attn.k_proj.in_features
    k_out = layer.self_attn.k_proj.out_features
    attn_type = "self-attn" if k_in > 500 else "cross-attn"
    n_params = sum(p.numel() for p in layer.parameters())
    print(f"{i:>4} {attn_type:<14} {k_in:>6} {k_out:>6} {n_params/1e6:>8.3f}M")

## 5. RMSNorm Count

In [ ]:
# Exact count for the proposed RMSNorm-only tuning improvement
norm_total = 0
print("RMSNORM PARAMETERS IN ACTION EXPERT")
print("=" * 60)
for name, param in model.named_parameters():
    if "vlm_with_expert.lm_expert" in name and "norm" in name.lower():
        norm_total += param.numel()
        print(f"  {name}: {param.shape} ({param.numel():,})")
print(f"\nTotal: {norm_total:,}")
# Decomposition: 16 layers x 2 norms x 720 + 1 final norm x 720 = 23,760

## 6. Vision Pipeline & Projection Layers

In [ ]:
# Connector (4x4 patch merge: Linear(12288, 960))
print("CONNECTOR")
print("=" * 60)
for name, param in model.named_parameters():
    if "connector" in name.lower():
        print(f"  {name}: {param.shape} ({param.numel():,})")

# SigLIP config
vision = model.model.vlm_with_expert.vlm.model.vision_model
vc = vision.config
print(f"\nSIGLIP CONFIG")
print("=" * 60)
for field in ["image_size", "patch_size", "hidden_size", "num_hidden_layers"]:
    print(f"  {field}: {getattr(vc, field, 'N/A')}")
print(f"  Patch embedding: {vision.embeddings.patch_embedding.weight.shape}")
# Pipeline: 256x256 input -> resize 512x512 -> 16x16 patches -> 1024 patches
#           -> 4x4 merge -> 64 tokens -> Linear(12288, 960)

In [ ]:
# Projection layers (action/state/time — all trainable)
print("PROJECTION LAYERS")
print("=" * 60)
proj_total = 0
for name, param in model.named_parameters():
    is_proj = any(k in name.lower() for k in [
        "action_in", "action_out", "state_proj", "time_mlp", "noise_proj",
    ])
    if is_proj or ("proj" in name.lower()
                   and "vlm_with_expert.vlm" not in name
                   and "lm_expert" not in name):
        print(f"  {name}: {param.shape} ({param.numel():,})")
        proj_total += param.numel()
print(f"\nTotal: {proj_total:,} ({proj_total/1e6:.3f}M)")

## 7. Reference Model Comparison

In [ ]:
# The reference checkpoint differs from the base in several critical ways
ref_model = SmolVLAPolicy.from_pretrained("HuggingFaceVLA/smolvla_libero")
ref_config = ref_model.config

print("BASE vs REFERENCE CONFIG")
print("=" * 60)
for field in [
    "vlm_model_name", "chunk_size", "n_action_steps", "max_action_dim",
    "num_steps", "expert_width_multiplier", "num_vlm_layers",
    "n_obs_steps", "self_attn_every_n_layers", "resize_imgs_with_padding",
    "attention_mode",
]:
    bv = getattr(config, field, "N/A")
    rv = getattr(ref_config, field, "N/A")
    tag = "OK" if str(bv) == str(rv) else "DIFFERS"
    print(f"  {field:<32} [{tag}]")
    if tag == "DIFFERS":
        print(f"    base: {bv}")
        print(f"    ref:  {rv}")

# Reference was trained directly on LIBERO (no cross-embodiment gap)
print(f"\nReference input/output features:")
for k, v in ref_config.input_features.items():
    print(f"  {k}: {v}")
for k, v in ref_config.output_features.items():
    print(f"  {k}: {v}")

del ref_model
torch.cuda.empty_cache()

## 8. Save Verification JSON

In [ ]:
verification = {
    "total_params_M": round(total_all / 1e6, 2),
    "trainable_params_M": round(trainable_all / 1e6, 2),
    "trainable_pct": round(100 * trainable_all / total_all, 1),
    "expert_norm_params": norm_total,
    "spatial_episodes": len(spatial_eps),
    "mean_episode_length": round(float(np.mean(spatial_lengths)), 1),
    "min_episode_length": int(min(spatial_lengths)),
    "max_episode_length": int(max(spatial_lengths)),
    "action_dim": int(actions.shape[-1]),
    "num_tasks": len(task_counts),
    "task_names": sorted(task_counts.keys()),
}

save_path = os.path.join(FIGURE_DIR, "..", "verification_results.json")
with open(save_path, "w") as f:
    json.dump(verification, f, indent=2, default=str)
print(f"Saved: {save_path}")
print(json.dumps(verification, indent=2, default=str))

del model
torch.cuda.empty_cache()
print("Done.")